# NumPy — fast arrays for physics

**What this library does in one sentence:** NumPy gives Python a fast, fixed-type array (think "a list of numbers that knows it's numbers") plus the math functions to operate on those arrays element-wise.

In this project you'll use NumPy for: state vectors `[x, ẋ, θ, θ̇]`, time arrays, trig in the equations of motion, and saving trajectories to disk.

---

## 1. Why lists aren't enough for physics

A Python `list` can hold anything — numbers, strings, other lists. That flexibility costs speed and ergonomics. To compute `sin(angle)` for 10,000 angles, with a list you'd loop. With a NumPy array, you write `np.sin(angles)` and it's done — in C, vectorised.

NumPy arrays:
- store one fixed numeric type (all `float64`, all `int32`, etc.)
- run loops in compiled C, not in Python
- support element-wise math: `a + b`, `np.sin(a)`, `a * 2`

In [ ]:
import numpy as np   # the universal alias — always import as 'np'

## 2. Creating arrays

Four functions cover 95% of array creation.

### `np.array(object)`

Convert a list (or list of lists) into a NumPy array.

| Parameter | Meaning |
|-----------|---------|
| `object`  | A Python list, tuple, or nested list to convert. |
| `dtype`   | (optional) Force a type, e.g. `dtype=float`. |

In [ ]:
# minimal example
a = np.array([1, 2, 3])
print(a, a.dtype)

# pendulum example — a state vector [x, x_dot, theta, theta_dot]
state = np.array([0.0, 0.0, 0.1, 0.0])   # cart at origin, pole leaning 0.1 rad
print(state)

**What this means for our project:** every state in our simulation is one 4-element `np.array`. The RK4 integrator advances `state` by adding scaled derivatives — `state + k1 * dt/2` — which only works because arrays support element-wise arithmetic.

### `np.zeros(shape)` and `np.ones(shape)`

Allocate an array filled with zeros (or ones). Used when you want a buffer to fill in later.

| Parameter | Meaning |
|-----------|---------|
| `shape`   | Integer (1D) or tuple (2D+), e.g. `5` or `(3, 4)`. |

In [ ]:
# minimal
print(np.zeros(5))         # [0. 0. 0. 0. 0.]
print(np.ones((2, 3)))     # 2 rows, 3 cols of ones

# pendulum — preallocate space for 1000 simulation steps
N = 1000
states = np.zeros((N, 4))  # 1000 rows, 4 columns (one row per timestep)
states[0] = [0.0, 0.0, 0.1, 0.0]
print(states.shape)

**Why preallocate?** Appending to a NumPy array is slow (it reallocates the whole buffer). For a simulation where you know the length, allocate once and write to `states[i]`.

### `np.linspace(start, stop, num)`

Return `num` evenly-spaced points between `start` and `stop`, **inclusive of both endpoints**.

| Parameter  | Meaning |
|------------|---------|
| `start`    | First value in the array. |
| `stop`     | Last value (included). |
| `num`      | How many points total — *not* the step size. |

**`num` vs step:** `linspace(0, 10, 5)` gives 5 points; the step is `(10-0)/(5-1) = 2.5`. If you want to control the step, use `arange` instead.

In [ ]:
# minimal
print(np.linspace(0, 1, 5))     # [0.   0.25 0.5  0.75 1.  ]

# pendulum — a time array for a 5-second simulation at 100 Hz
T_END, FPS = 5.0, 100
t = np.linspace(0, T_END, int(T_END * FPS) + 1)
print(t.shape, t[0], t[-1])

**For us:** `linspace` is what we use whenever we want a time axis for plotting. Pair it with the saved state array and you have `(t, states)`, the universal output format of a simulation.

### `np.arange(start, stop, step)`

Like Python's `range`, but returns an array and accepts float `step`. **`stop` is exclusive.**

| Parameter | Meaning |
|-----------|---------|
| `start`   | First value. |
| `stop`    | Stops *before* this value. |
| `step`    | Increment between consecutive values. |

In [ ]:
# minimal
print(np.arange(0, 1, 0.25))   # [0.   0.25 0.5  0.75]  — note: 1.0 not included

# pendulum — angle sweep for a phase portrait grid
angles_deg = np.arange(-60, 61, 10)     # -60, -50, ..., 50, 60
print(angles_deg)

**Rule of thumb:** if you care about "how many points," use `linspace`. If you care about "step size," use `arange`. For time arrays we almost always want `linspace`.

## 3. Array operations

### Element-wise arithmetic

The operators `+ - * /` act on each element individually, no loops needed.

In [ ]:
# minimal
a = np.array([1, 2, 3])
b = np.array([10, 20, 30])
print(a + b)        # [11 22 33]
print(a * b)        # [10 40 90]  — element-wise, NOT matrix multiplication
print(b / a)        # [10. 10. 10.]

# pendulum — RK4 combines four slopes element-wise
k1 = np.array([0.0, 0.5, 0.3, -1.2])
k2 = np.array([0.5, 0.7, 0.4, -1.1])
dt = 0.01
mid_state = k1 + k2 * dt / 2
print(mid_state)

### Broadcasting

When array shapes don't match, NumPy *broadcasts* the smaller one across the larger. The classic case: scalar × array.

In [ ]:
# scalar broadcasts to every element
a = np.array([1, 2, 3])
print(a * 2)        # [2 4 6]      — 2 broadcast across all 3 elements
print(a + 10)       # [11 12 13]

# row + column broadcasts to a grid
row = np.array([1, 2, 3])         # shape (3,)
col = np.array([[10], [20]])      # shape (2, 1)
print(row + col)                  # shape (2, 3)

**Why we care:** to convert a whole array of radians to degrees, we write `np.degrees(angles)` — no loop. To scale a force array, `forces * mass`. Broadcasting is what makes vectorised code short.

### `np.dot(a, b)`

Dot product (vectors) or matrix multiplication (matrices). Distinct from `a * b`, which is element-wise.

| Parameter | Meaning |
|-----------|---------|
| `a`, `b`  | The two arrays. Shapes must align: `(n,) · (n,)` for a scalar; `(m,n) · (n,p)` for `(m,p)`. |

In [ ]:
# minimal
u = np.array([1, 2, 3])
v = np.array([4, 5, 6])
print(np.dot(u, v))      # 1*4 + 2*5 + 3*6 = 32

# pendulum — LQR control law:  F = -K @ state
K     = np.array([1.0, 2.0, 30.0, 5.0])     # gain (computed later from python-control)
state = np.array([0.05, 0.0, 0.1, 0.0])
F     = -K @ state                          # @ is the modern Python operator for np.dot
print(F)

**Note the `@` operator:** since Python 3.5, `a @ b` is equivalent to `np.dot(a, b)` and reads better. Use it for matrix-vector products.

## 4. Math functions

NumPy's math functions take arrays and return arrays. They're the vectorised replacements for `math.sin`, `math.sqrt`, etc.

### `np.sin(x)`, `np.cos(x)`, `np.tan(x)`

Trigonometric functions. **Argument is always in radians.**

In [ ]:
# minimal — sine of three angles at once
angles = np.array([0, np.pi/2, np.pi])
print(np.sin(angles))         # [0. 1. 0.] (tiny float noise on pi)

# pendulum — the EOM contains sin(theta) and cos(theta) terms
theta = 0.1            # ~5.7 degrees from vertical
print(np.sin(theta), np.cos(theta))

### `np.radians(deg)` and `np.degrees(rad)`

Convert between degrees and radians. The simulation runs in radians but humans think in degrees, so we convert at the boundary (input + display).

In [ ]:
# minimal
print(np.radians(180))        # 3.141592653589793
print(np.degrees(np.pi/4))    # 45.0

# pendulum — set the initial angle in human-readable degrees
THETA_INIT = np.radians(5)    # 5 degrees → ~0.0873 rad
state = np.array([0.0, 0.0, THETA_INIT, 0.0])

# ...and convert back for display
print(f"theta = {np.degrees(state[2]):.1f} deg")

### `np.sqrt(x)`, `np.abs(x)`, `np.exp(x)`

Square root, absolute value, exponential. All vectorised.

In [ ]:
# minimal
print(np.sqrt([1, 4, 9]))     # [1. 2. 3.]
print(np.abs([-2, 3, -4]))    # [2 3 4]
print(np.exp([0, 1]))         # [1.       2.71828]

# pendulum — energy:  E = 0.5 * m * v^2  +  m * g * h
m, g, L = 0.3, 9.81, 0.8
v, theta = 1.2, 0.1
E = 0.5 * m * v**2 + m * g * L * (1 - np.cos(theta))
print(f"E = {E:.4f} J")

## 5. Array inspection

### `.shape` and `.dtype`

Attributes (no parentheses) that tell you the dimensions and type of an array.

In [ ]:
a = np.zeros((100, 4))
print(a.shape)     # (100, 4)  — 100 rows, 4 columns
print(a.dtype)     # float64

# pendulum — sanity-check the trajectory shape after a simulation
states = np.zeros((500, 4))
assert states.shape == (500, 4), "trajectory must be (timesteps, 4)"

### `np.min(a)` and `np.max(a)`

Minimum and maximum of an array.

| Parameter | Meaning |
|-----------|---------|
| `a`       | The array. |
| `axis`    | (optional) `0` for column-wise, `1` for row-wise; default = whole array. |

In [ ]:
# minimal
a = np.array([3, 1, 4, 1, 5, 9, 2, 6])
print(np.min(a), np.max(a))       # 1 9

# pendulum — peak angle reached during the simulation
theta_history = np.radians([5, 8, 12, 9, 3, -2, -8, -11, -6, 1])
peak_deg = np.degrees(np.max(np.abs(theta_history)))
print(f"peak deflection = {peak_deg:.1f} deg")

### Indexing and slicing

Like Python lists, but extended to multiple dimensions with commas.

In [ ]:
states = np.array([
    [0.00, 0.0, 0.10, 0.0],
    [0.01, 0.5, 0.11, 0.2],
    [0.03, 0.9, 0.13, 0.3],
    [0.05, 1.2, 0.16, 0.4],
])

# single element  (row, col)
print(states[0, 2])      # theta at timestep 0 = 0.10

# whole row  (one timestep)
print(states[1])         # [0.01 0.5  0.11 0.2 ]

# whole column  (one variable over all time)
print(states[:, 2])      # theta over all timesteps = [0.10 0.11 0.13 0.16]

# slice — first 2 timesteps
print(states[:2])

**For us:** after running the simulation, `states[:, 2]` extracts the entire `theta` trajectory in one line. That's how we feed `(time, theta)` to matplotlib for a plot.

## 6. Saving and loading data

Simulations are expensive. Run once, save to disk, then load for plotting / Manim animation / further analysis.

### `np.savez(filename, key=array, ...)`

Save several arrays into a single compressed file as named keys.

| Parameter   | Meaning |
|-------------|---------|
| `filename`  | Output file. Extension `.npz` is added automatically. |
| `key=array` | Pass each array as a keyword argument. The key is how you'll retrieve it. |

In [ ]:
# pendulum — save a complete simulation trajectory
t      = np.linspace(0, 5, 501)
states = np.zeros((501, 4))                # placeholder — would be filled by RK4
states[:, 2] = 0.1 * np.cos(2.0 * t)       # fake oscillation for the demo

np.savez('pendulum_data.npz',
         t=t,
         states=states,
         params=np.array([1.0, 0.3, 0.8, 9.81]))   # M, m, L, g
print('saved')

### `np.load(filename)`

Read a `.npz` file. The return value behaves like a dictionary — access arrays by their key.

In [ ]:
data   = np.load('pendulum_data.npz')
t      = data['t']
states = data['states']
params = data['params']

print(t.shape, states.shape)
print(f"loaded params: M={params[0]}  m={params[1]}  L={params[2]}  g={params[3]}")

### Why `.npz` over CSV?

| | CSV | NPZ |
|--|------|------|
| stores multiple arrays? | no (one table) | yes (named keys) |
| preserves dtype? | no (everything is text) | yes (`float64` stays `float64`) |
| handles multi-dimensional? | awkward | native |
| file size? | larger (text) | smaller (binary, compressed) |
| load speed? | slow (parse text) | fast (memory-map) |

For simulation data — multi-column trajectories, parameter dicts, multiple runs in one file — `.npz` is the right call. Save CSV only when something *outside Python* needs to read it (e.g. Excel).

## Functions used in this project — quick reference

| Function | Signature | What it does |
|----------|-----------|--------------|
| `np.array`         | `np.array(object)`                  | Convert a list to an array.                              |
| `np.zeros`         | `np.zeros(shape)`                   | Array of zeros with the given shape.                     |
| `np.ones`          | `np.ones(shape)`                    | Array of ones.                                           |
| `np.linspace`      | `np.linspace(start, stop, num)`     | `num` evenly-spaced values from `start` to `stop`.       |
| `np.arange`        | `np.arange(start, stop, step)`      | Values from `start` (incl) to `stop` (excl) in `step`s.  |
| `np.dot` / `@`     | `np.dot(a, b)` or `a @ b`           | Dot product / matrix multiplication.                     |
| `np.sin/cos/tan`   | `np.sin(x)`                         | Trig — argument in radians.                              |
| `np.radians`       | `np.radians(deg)`                   | Degrees → radians.                                       |
| `np.degrees`       | `np.degrees(rad)`                   | Radians → degrees.                                       |
| `np.sqrt`          | `np.sqrt(x)`                        | Element-wise square root.                                |
| `np.abs`           | `np.abs(x)`                         | Element-wise absolute value.                             |
| `np.exp`           | `np.exp(x)`                         | Element-wise e^x.                                        |
| `.shape`           | `a.shape`                           | Tuple of array dimensions.                               |
| `.dtype`           | `a.dtype`                           | Element type (e.g. `float64`).                           |
| `np.min` / `np.max`| `np.min(a)` / `np.max(a)`           | Smallest / largest element.                              |
| indexing           | `a[i, j]`, `a[:, k]`, `a[:n]`       | Element / column / slice access.                         |
| `np.savez`         | `np.savez(file, k1=a1, k2=a2, ...)` | Save named arrays to a compressed `.npz` file.           |
| `np.load`          | `np.load(file)`                     | Load `.npz`; access via `data['key']`.                   |